# SmartLens-Indic recognition training (IndicSTR12 Hindi word images)

This notebook uses the prepared recognition manifests only. It does not modify the source ZIP archives or the committed split files. Training, evaluation, and GPU results have not been run from this notebook.

**Important:** the official `devanagari_PP-OCRv5_mobile_rec.yaml` recipe has an empty `Global.pretrained_model`. Its supported path here is therefore training from scratch, not fine-tuning. The inference `.pdiparams` file is not a PaddleOCR training checkpoint and is not used as one.

This notebook selects official PaddlePaddle 3.3.0 GPU wheels. The current Windows app environment is PaddlePaddle 3.0.0 CPU / PaddleOCR 3.7.0 / PaddleX 3.7.2. The Colab smoke test is not a substitute for loading the copied model in that exact Windows environment.

## Before running

One-time shared transfer: in Google Drive create `MyDrive/SmartLens-Indic/Data/` and copy both original `Smartlens Dataset-*.zip` archives there once, without extracting or editing them. Reuse these same two archive copies for both training notebooks; do not upload them again for detection. For this notebook, also copy `dataset_work/recognition/train.csv`, `validation.csv`, and `test.csv` into `MyDrive/SmartLens-Indic/dataset_work/recognition/`. The notebook reads these files and writes extracted working images only under temporary `/content/`; checkpoints, logs, metrics, and exported models go to Drive.

The prepared labeled split counts must be 15,901 / 2,044 / 2,055. The separate 5,000-image unlabelled source test list is not used as supervised data.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

PROJECT = Path('/content/drive/MyDrive/SmartLens-Indic')
ARCHIVE_DIR = PROJECT / 'Data'
MANIFEST_DIR = PROJECT / 'dataset_work' / 'recognition'
ARCHIVES = {path.name: path for path in ARCHIVE_DIR.glob('Smartlens Dataset-*.zip')}
assert len(ARCHIVES) == 2, f'Expected both original ZIPs under {ARCHIVE_DIR}; found {len(ARCHIVES)}'
for split in ('train', 'validation', 'test'):
    assert (MANIFEST_DIR / f'{split}.csv').is_file(), f'Missing {MANIFEST_DIR / (split + ".csv")}'
print('Drive inputs found:', *ARCHIVES.keys(), sep='\n- ')
print('Original ZIPs will be opened read-only; extraction target is /content only.')

In [ ]:
# Check the runtime before installing anything. Paddle's official Linux wheel docs list Python 3.9-3.13.
import platform, re, subprocess, sys
assert (3, 9) <= sys.version_info[:2] <= (3, 13), f'Unsupported Python: {platform.python_version()}'
gpu_info = subprocess.run(['nvidia-smi'], check=True, text=True, capture_output=True).stdout
print(gpu_info.splitlines()[0])
match = re.search(r'CUDA Version:\s*(\d+\.\d+)', gpu_info)
assert match, 'Could not detect the driver CUDA version; stop and inspect the Colab runtime.'
cuda_version = float(match.group(1))
if cuda_version >= 13.0:
    wheel = 'cu130'
elif cuda_version >= 12.9:
    wheel = 'cu129'
elif cuda_version >= 12.6:
    wheel = 'cu126'
elif cuda_version >= 11.8:
    wheel = 'cu118'
else:
    raise RuntimeError(f'No verified Paddle 3.3.0 wheel selected for CUDA {cuda_version}')
print('Selected official Paddle wheel index:', wheel)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'paddlepaddle', 'paddlepaddle-gpu'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'paddlepaddle-gpu==3.3.0', '-i', f'https://www.paddlepaddle.org.cn/packages/stable/{wheel}/'], check=True)
import paddle
paddle.set_device('gpu:0')
assert paddle.is_compiled_with_cuda(), 'Installed Paddle is CPU-only; stop before training.'
assert paddle.device.get_device().startswith('gpu'), paddle.device.get_device()
print('Python:', platform.python_version(), '| Paddle:', paddle.__version__, '| Device:', paddle.device.get_device())

In [ ]:
# Pin the official training code and install only its declared training requirements.
import subprocess
PPOCR_ROOT = Path('/content/PaddleOCR')
PPOCR_COMMIT = 'b03f46425e8ff4442b268ce449e3eef758146cd4'  # PaddleOCR v3.7.0
if not PPOCR_ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/PaddlePaddle/PaddleOCR.git', str(PPOCR_ROOT)], check=True)
subprocess.run(['git', '-C', str(PPOCR_ROOT), 'checkout', PPOCR_COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(PPOCR_ROOT / 'requirements.txt')], check=True)
print('PaddleOCR source revision:', subprocess.check_output(['git', '-C', str(PPOCR_ROOT), 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
# Validate the untouched manifests, official dictionary, group split, labels, and ZIP members before extraction.
import csv, hashlib, zipfile, cv2, numpy as np, unicodedata
EXPECTED = {'train': 15901, 'validation': 2044, 'test': 2055}
rows_by_split = {}
group_owner, image_owner = {}, {}
for split, expected_count in EXPECTED.items():
    with (MANIFEST_DIR / f'{split}.csv').open(encoding='utf-8', newline='') as handle:
        rows = list(csv.DictReader(handle))
    assert len(rows) == expected_count, f'{split}: expected {expected_count}, got {len(rows)}'
    rows_by_split[split] = rows
    for row in rows:
        assert row['split'] == split and row['transcription'].strip()
        assert any('\u0900' <= ch <= '\u097f' for ch in row['transcription']), row['image_path']
        for key in ('group_id', 'sha256'):
            owners = group_owner if key == 'group_id' else image_owner
            old = owners.setdefault(row[key], split)
            assert old == split, f'{key} leaks across splits: {row[key]}'
dict_path = PPOCR_ROOT / 'ppocr' / 'utils' / 'dict' / 'ppocrv5_devanagari_dict.txt'
characters = set(dict_path.read_text(encoding='utf-8').splitlines())
unknown = sorted({ch for rows in rows_by_split.values() for row in rows for ch in row['transcription'] if ch != ' ' and ch not in characters})
max_label_length = max(len(row['transcription']) for rows in rows_by_split.values() for row in rows)
assert not unknown, f'Labels contain characters absent from official dictionary: {unknown[:20]}'
assert max_label_length <= 25, f'Label length {max_label_length} exceeds official max_text_length=25'
print('Verified row counts:', {key: len(value) for key, value in rows_by_split.items()})
print('Official dictionary coverage: complete; maximum label length:', max_label_length)
print('Source-group and exact-image hashes are split-disjoint.')

In [ ]:
# Materialize each referenced member into ephemeral Colab storage and create the official TSV label lists.
TRAINING_DATA_VALIDATED = False
import hashlib, os
WORK = Path('/content/smartlens_recognition')
REC_DATA = WORK / 'train_data'
LIST_DIR = WORK / 'labels'
REC_DATA.mkdir(parents=True, exist_ok=True)
LIST_DIR.mkdir(parents=True, exist_ok=True)
zip_handles = {name: zipfile.ZipFile(path, 'r') for name, path in ARCHIVES.items()}
list_paths = {}
try:
    for split, rows in rows_by_split.items():
        list_path = LIST_DIR / f'{split}.txt'
        with list_path.open('w', encoding='utf-8', newline='') as output:
            for row in rows:
                archive_name = Path(row['archive_path'].replace('\\', '/')).name
                assert archive_name in zip_handles, f'Archive not transferred: {archive_name}'
                member = row['archive_member']
                raw = zip_handles[archive_name].read(member)
                assert hashlib.sha256(raw).hexdigest() == row['sha256'], f'Image bytes do not match manifest hash: {member}'
                suffix = Path(member).suffix.lower() or '.png'
                name = hashlib.sha256(member.encode('utf-8')).hexdigest()[:20] + suffix
                target = REC_DATA / 'images' / split / name
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(raw)
                decoded = cv2.imdecode(np.frombuffer(raw, dtype=np.uint8), cv2.IMREAD_COLOR)
                assert decoded is not None and decoded.size, f'Unreadable image: {member}'
                relative = target.relative_to(REC_DATA).as_posix()
                output.write(f'{relative}\t{row["transcription"]}\n')
        list_paths[split] = list_path
finally:
    for handle in zip_handles.values():
        handle.close()
print('Working images and label lists created under', WORK)
print('The Drive ZIP files and split manifests were not written to.')
TRAINING_DATA_VALIDATED = True

In [ ]:
# Create a separate config copy. Architecture, dictionary, 75 epochs, and max_text_length stay official.
import yaml
OFFICIAL_CONFIG = PPOCR_ROOT / 'configs/rec/PP-OCRv5/multi_language/devanagari_PP-OCRv5_mobile_rec.yaml'
TRAIN_CONFIG = WORK / 'devanagari_rec_training.yml'
CHECKPOINT_DIR = PROJECT / 'training' / 'recognition' / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
config = yaml.safe_load(OFFICIAL_CONFIG.read_text(encoding='utf-8'))
assert config['Global']['pretrained_model'] in (None, ''), 'Unexpected official pretrained checkpoint; re-verify recipe.'
config['Global']['save_model_dir'] = str(CHECKPOINT_DIR)
config['Global']['save_epoch_step'] = 1
config['Global']['character_dict_path'] = str(dict_path)
config['Global']['pretrained_model'] = None
config['Global']['checkpoints'] = None
config['Train']['dataset']['data_dir'] = str(REC_DATA)
config['Train']['dataset']['label_file_list'] = [str(list_paths['train'])]
config['Eval']['dataset']['data_dir'] = str(REC_DATA)
config['Eval']['dataset']['label_file_list'] = [str(list_paths['validation'])]
TRAIN_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding='utf-8')
print('Training config copy:', TRAIN_CONFIG)
print('This recipe trains from scratch; no recognized training checkpoint exists in the official config.')

## Training (expensive; run only after all checks above pass)

The official recipe has no pretrained training checkpoint and saves complete `latest` checkpoints to Drive after every epoch. The training cell requires successful dataset validation/materialization. After a Colab interruption, remount Drive and rerun the setup and validation cells; the training cell automatically resumes only when `latest.pdparams`, `latest.pdopt`, and `latest.states` are all present. It stops for an incomplete checkpoint instead of silently starting over. Console logs are appended in the same Drive checkpoint folder. This cell is intentionally not run during notebook preparation.

In [ ]:
import subprocess
assert TRAINING_DATA_VALIDATED is True, 'Run and pass dataset validation/materialization before training.'
assert set(list_paths) == {'train', 'validation', 'test'}
assert all(path.is_file() for path in list_paths.values()), 'One or more validated label lists are missing.'
checkpoint_prefix = CHECKPOINT_DIR / 'latest'
checkpoint_files = [Path(str(checkpoint_prefix) + suffix) for suffix in ('.pdparams', '.pdopt', '.states')]
existing_checkpoints = [checkpoint_file.is_file() for checkpoint_file in checkpoint_files]
if any(existing_checkpoints) and not all(existing_checkpoints):
    missing = [str(file) for file, exists in zip(checkpoint_files, existing_checkpoints) if not exists]
    raise RuntimeError(f'Incomplete latest checkpoint; missing {missing}. Preserve the Drive checkpoint folder and inspect it before continuing.')
RESUME = all(existing_checkpoints)
if RESUME:
    print('Resuming from complete Drive checkpoint:', checkpoint_prefix)
else:
    print('No complete latest checkpoint found; this is a fresh training run.')
command = [sys.executable, 'tools/train.py', '-c', str(TRAIN_CONFIG)]
if RESUME:
    command += ['-o', f'Global.checkpoints={checkpoint_prefix}']
log_path = CHECKPOINT_DIR / 'colab_train_console.log'
with log_path.open('a', encoding='utf-8') as log:
    process = subprocess.Popen(command, cwd=PPOCR_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    code = process.wait()
if code:
    raise RuntimeError(f'Training exited with code {code}; inspect {log_path} before resuming.')


In [ ]:
# Official validation/test evaluation. Test annotations are read only after training is complete.
TEST_CONFIG = WORK / 'devanagari_rec_test.yml'
test_config = yaml.safe_load(TRAIN_CONFIG.read_text(encoding='utf-8'))
test_config['Eval']['dataset']['label_file_list'] = [str(list_paths['test'])]
TEST_CONFIG.write_text(yaml.safe_dump(test_config, sort_keys=False, allow_unicode=True), encoding='utf-8')
checkpoint = CHECKPOINT_DIR / 'best_accuracy'
for split, cfg in (('validation', TRAIN_CONFIG), ('test', TEST_CONFIG)):
    command = [sys.executable, 'tools/eval.py', '-c', str(cfg), '-o', f'Global.checkpoints={checkpoint}']
    result = subprocess.run(command, cwd=PPOCR_ROOT, text=True, capture_output=True)
    log = PROJECT / 'training' / 'recognition' / f'{split}_official_eval.log'
    log.write_text(result.stdout + '\n' + result.stderr, encoding='utf-8')
    print(result.stdout[-5000:])
    result.check_returncode()
    print('Full log:', log)

## Export and model-layout compatibility check

The official exporter writes an inference config plus static inference weights. The app expects the PaddleX `TextRecognition(model_name=..., model_dir=...)` layout. This cell packages the exported files into that directory and runs a CPU smoke test. Then copy the model to Windows and repeat the same smoke test with PaddlePaddle 3.0.0 CPU / PaddleOCR 3.7.0 / PaddleX 3.7.2 before adoption. A Colab smoke test proves loadability only under Colab's runtime; it does not prove Windows compatibility or improved recognition accuracy.

In [ ]:
import shutil
EXPORT_ROOT = PROJECT / 'training' / 'recognition' / 'export'
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
EXPORT_CONFIG = WORK / 'devanagari_rec_export.yml'
export_config = yaml.safe_load(TRAIN_CONFIG.read_text(encoding='utf-8'))
export_config['Global']['checkpoints'] = str(checkpoint)
export_config['Global']['pretrained_model'] = None
export_config['Global']['save_inference_dir'] = str(EXPORT_ROOT)
EXPORT_CONFIG.write_text(yaml.safe_dump(export_config, sort_keys=False, allow_unicode=True), encoding='utf-8')
subprocess.run([sys.executable, 'tools/export_model.py', '-c', str(EXPORT_CONFIG)], cwd=PPOCR_ROOT, check=True)
WINDOWS_MODEL = PROJECT / 'training' / 'recognition' / 'windows_model'
WINDOWS_MODEL.mkdir(parents=True, exist_ok=True)
for file in (EXPORT_ROOT / 'inference').iterdir():
    if file.is_file(): shutil.copy2(file, WINDOWS_MODEL / file.name)
shutil.copy2(EXPORT_ROOT / 'inference.yml', WINDOWS_MODEL / 'inference.yml')
assert (WINDOWS_MODEL / 'inference.yml').is_file()
assert (WINDOWS_MODEL / 'inference.pdiparams').is_file(), 'Exported Paddle parameters are missing.'
assert (WINDOWS_MODEL / 'inference.json').is_file() or (WINDOWS_MODEL / 'inference.pdmodel').is_file(), 'Exported Paddle model graph is missing.'
subprocess.run([sys.executable, '-m', 'pip', 'install', 'paddleocr==3.7.0'], check=True)
from paddleocr import TextRecognition
sample_row = rows_by_split['test'][0]
sample_name = hashlib.sha256(sample_row['archive_member'].encode('utf-8')).hexdigest()[:20] + Path(sample_row['archive_member']).suffix.lower()
sample_path = REC_DATA / 'images' / 'test' / sample_name
predictor = TextRecognition(model_name='devanagari_PP-OCRv5_mobile_rec', model_dir=str(WINDOWS_MODEL), device='cpu')
prediction = predictor.predict(input=str(sample_path))
assert prediction, 'No prediction returned from exported model.'
print('CPU export smoke test result:', prediction[0].json)
print('Packaged model directory:', WINDOWS_MODEL)

In [ ]:
# Labeled IndicSTR12 test results: CER, WER, exact match, confidence, and per-image wall time.
import csv, time, unicodedata
def edit_distance(reference, hypothesis):
    previous = list(range(len(hypothesis) + 1))
    for i, ref_char in enumerate(reference, 1):
        current = [i]
        for j, hyp_char in enumerate(hypothesis, 1):
            current.append(min(current[-1] + 1, previous[j] + 1, previous[j - 1] + (ref_char != hyp_char)))
        previous = current
    return previous[-1]
def normalized(text):
    return ' '.join(unicodedata.normalize('NFC', text).split())
metric_path = PROJECT / 'training' / 'recognition' / 'test_per_image_metrics.csv'
metric_path.parent.mkdir(parents=True, exist_ok=True)
records = []
for row in rows_by_split['test']:
    name = hashlib.sha256(row['archive_member'].encode('utf-8')).hexdigest()[:20] + Path(row['archive_member']).suffix.lower()
    image_path = REC_DATA / 'images' / 'test' / name
    started = time.perf_counter()
    result = predictor.predict(input=str(image_path))[0].json['res']
    elapsed_ms = (time.perf_counter() - started) * 1000
    reference = normalized(row['transcription'])
    hypothesis = normalized(result['rec_text'])
    ref_words, hyp_words = reference.split(), hypothesis.split()
    records.append({'image_path': row['image_path'], 'reference_text': reference, 'recognized_text': hypothesis,
                    'cer': edit_distance(list(reference), list(hypothesis)) / max(1, len(reference)),
                    'wer': edit_distance(ref_words, hyp_words) / max(1, len(ref_words)),
                    'exact_match': reference == hypothesis, 'confidence': result.get('rec_score'),
                    'confidence_type': 'PaddleOCR model score; not calibrated probability', 'processing_ms': round(elapsed_ms, 2)})
with metric_path.open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=records[0].keys(), lineterminator='\n')
    writer.writeheader(); writer.writerows(records)
print('Test images:', len(records))
print('Mean CER:', sum(item['cer'] for item in records) / len(records))
print('Mean WER:', sum(item['wer'] for item in records) / len(records))
print('Exact match:', sum(item['exact_match'] for item in records), '/', len(records))
print('Per-image report:', metric_path)

## Reported measures and limitations

The official recognizer evaluation reports its configured recognition metrics. Do not equate confidence with accuracy. Compare CER, WER, and exact-match against the prepared verified references using a separate per-image evaluation script before reporting recognition quality. Keep these word-image results separate from scene-image results. No result should be claimed until the GPU training, evaluation, export, and Windows CPU smoke test have actually completed.